# MIDO AI 모델 테스트 및 평가 노트북

이 노트북은 MIDO 프로젝트의 AI 모델(머신러닝 악용 위험 예측, 딥러닝 심각도 추정) 및 취약점 분석 파이프라인을 테스트하고 평가합니다.

1. **데이터 적재 및 탐색**: 정제된 CVE 데이터셋, KEV, CWE, OWASP 통계 확인
2. **피처 엔지니어링 & CVSS 계산**: CVSS v3.1 벡터 파싱 및 공식 점수 산출
3. **ML 모델 테스트**: RandomForest 악용 위험 예측 (`exploit_rf.joblib`)
4. **DL 모델 테스트**: PyTorch MLP 설명문 심각도 분류 (`severity_mlp.pt`)
5. **통합 엔드투엔드 분석 파이프라인 검증**: 의존성 매니페스트 분석 및 우선순위 리포트 생성


In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

# 프로젝트 루트 및 ai 경로 설정
PROJECT_ROOT = Path("..").resolve()
AI_DIR = PROJECT_ROOT / "ai"
sys.path.insert(0, str(AI_DIR))

print(f"Project Root: {PROJECT_ROOT}")
print(f"AI Directory: {AI_DIR}")


Project Root: /Users/jeong-yelim/.gemini/antigravity/scratch/mIdo_v2
AI Directory: /Users/jeong-yelim/.gemini/antigravity/scratch/mIdo_v2/ai


## 1. 데이터 적재 및 기본 통계 확인
전처리 완료된 공공데이터(`data/processed/`)를 확인합니다.


In [2]:
processed_dir = PROJECT_ROOT / "data" / "processed"

# 1) 전체 CVE 데이터셋
df_cve = pd.read_csv(processed_dir / "cve_dataset.csv")
print(f"CVE 데이터셋 크기: {df_cve.shape} (행, 열)")
print(f"악용 라벨 분포:\n{df_cve['exploit_label'].value_counts(dropna=False)}")
df_cve[["cve_id", "cvss_score", "severity", "epss", "in_kev", "owasp", "exploit_label"]].head()


CVE 데이터셋 크기: (80636, 15) (행, 열)
악용 라벨 분포:
exploit_label
0    78717
1     1919
Name: count, dtype: int64


,cve_id,cvss_score,severity,epss,in_kev,owasp,exploit_label
0,CVE-2025-11570,4.6,MEDIUM,0.00218,0,A05,0
1,CVE-2025-61871,6.7,MEDIUM,0.00168,0,OTHER,0
2,CVE-2025-10124,4.5,MEDIUM,0.00264,0,OTHER,0
3,CVE-2025-21044,4.4,MEDIUM,0.00125,0,OTHER,0
4,CVE-2025-21045,5.5,MEDIUM,0.00125,0,A01,0


In [3]:
# 2) KEV & CWE & OWASP 통계
df_kev = pd.read_csv(processed_dir / "kev_clean.csv")
df_cwe = pd.read_csv(processed_dir / "cwe_clean.csv")
df_owasp = pd.read_csv(processed_dir / "owasp_stats.csv")

print(f"CISA KEV 등재 CVE 수: {len(df_kev)}")
print(f"정제된 CWE 목록 수: {len(df_cwe)}")
print("\nOWASP Top 10 분류별 통계:")
df_owasp


CISA KEV 등재 CVE 수: 1739
정제된 CWE 목록 수: 944

OWASP Top 10 분류별 통계:


,owasp,owasp_name,cve_count,avg_cvss,avg_epss,kev_rate,exploit_rate
0,A01,접근 제어 실패,20635,6.5561,0.0111,0.0117,0.0126
1,A02,보안 설정 오류,333,6.9670,0.0217,0.0240,0.0330
2,A03,소프트웨어 공급망 실패,35,7.0914,0.0061,0.0000,0.0000
3,A04,암호화 실패,1073,6.9395,0.0097,0.0121,0.0130
4,A05,인젝션,20200,7.1511,0.0192,0.0192,0.0238
5,A06,안전하지 않은 설계,4162,7.3109,0.0125,0.0171,0.0216
6,A07,인증 실패,4575,7.6459,0.0232,0.0278,0.0302
7,A08,소프트웨어·데이터 무결성 실패,1752,8.0973,0.0361,0.0457,0.0508
8,A09,보안 로깅·경보 실패,300,5.6897,0.0037,0.0067,0.0100
9,A10,예외 상황 처리 미흡,1570,6.1762,0.0068,0.0076,0.0083


## 2. 피처 엔지니어링 및 CVSS v3.1 공식 계산 테스트
`src.scoring` 모듈의 CVSS 벡터 파싱 및 점수 계산 수식을 검증합니다.


In [4]:
from src.scoring import cvss3_base_score, parse_cvss_vector, severity_from_score, make_ml_frame

# Log4Shell (CVE-2021-44228) 벡터 테스트
log4shell_vector = "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:H/A:H"
parts = parse_cvss_vector(log4shell_vector)
score = cvss3_base_score(log4shell_vector)
sev = severity_from_score(score)

print(f"벡터: {log4shell_vector}")
print(f"파싱 결과: {parts}")
print(f"계산된 CVSS 기본 점수: {score} (예상: 10.0)")
print(f"산출된 심각도: {sev} (예상: CRITICAL)")

# DataFrame 피처 변환 테스트
sample_rows = [
    {"cvss_vector": log4shell_vector, "cvss_score": score, "owasp": "A05"},
    {"cvss_vector": "CVSS:3.1/AV:L/AC:H/PR:H/UI:R/S:U/C:L/I:N/A:N", "cvss_score": 1.8, "owasp": "OTHER"}
]
feature_df = make_ml_frame(sample_rows)
print("\n피처 변환 결과:")
feature_df


벡터: CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:H/A:H
파싱 결과: {'attack_vector': 'N', 'attack_complexity': 'L', 'privileges': 'N', 'user_interaction': 'N', 'scope': 'C', 'confidentiality': 'H', 'integrity': 'H', 'availability': 'H'}
계산된 CVSS 기본 점수: 10.0 (예상: 10.0)
산출된 심각도: CRITICAL (예상: CRITICAL)

피처 변환 결과:


,attack_vector,attack_complexity,privileges,user_interaction,scope,confidentiality,integrity,availability,owasp,cvss_score
0,N,L,N,N,C,H,H,H,A05,10.0
1,L,H,H,R,U,L,N,N,OTHER,1.8


## 3. 머신러닝 악용 위험 예측 모델 테스트 (RandomForest)
`src.inference.exploit_ml`을 사용하여 실제 악용될 확률을 추론합니다.


In [5]:
from src.inference import load_ml_model, predict_exploit

ml_model = load_ml_model()
print("ML 모델 로드 완료:", type(ml_model))

# 샘플 데이터로 악용 위험 예측
exploit_probabilities = predict_exploit(feature_df)
for row, prob in zip(sample_rows, exploit_probabilities):
    print(f"OWASP: {row['owasp']}, CVSS: {row['cvss_score']} -> 악용 위험 확률: {prob:.4f} ({prob*100:.1f}%)")


ML 모델 로드 완료: <class 'sklearn.pipeline.Pipeline'>
OWASP: A05, CVSS: 10.0 -> 악용 위험 확률: 0.8518 (85.2%)
OWASP: OTHER, CVSS: 1.8 -> 악용 위험 확률: 0.1581 (15.8%)


## 4. 딥러닝 설명문 기반 심각도 추론 모델 테스트 (PyTorch MLP)
`src.inference.severity_dl`을 사용하여 취약점 텍스트 설명에서 심각도(LOW, MEDIUM, HIGH, CRITICAL)를 분류합니다.


In [6]:
from src.inference import load_dl_model, predict_severity

dl_model, vectorizer = load_dl_model()
print("DL 모델 및 TF-IDF 벡터라이저 로드 완료:")
print(f" - PyTorch 모델: {type(dl_model)}")
print(f" - 입력 단어 사전 크기: {len(vectorizer.vocabulary_)}단어")

test_descriptions = [
    "Remote code execution vulnerability allowing unauthenticated attacker to take full control of the server via malicious payload injection.",
    "Information disclosure flaw where internal server paths and debug logs are leaked to unauthorized users in error response.",
    "Denial of service issue causing high CPU usage when parsing specially crafted XML documents.",
    "Minor input validation issue in administrative panel with limited privileges required."
]

dl_predictions = predict_severity(test_descriptions)
for text, pred in zip(test_descriptions, dl_predictions):
    print(f"설명: {text[:60]}...")
    print(f" -> 예측 심각도: {pred['label']}, 신뢰도 확률: {pred['probability']:.4f}\n")


DL 모델 및 TF-IDF 벡터라이저 로드 완료:
 - PyTorch 모델: <class 'src.inference.severity_dl.SeverityMLP'>
 - 입력 단어 사전 크기: 4000단어
설명: Remote code execution vulnerability allowing unauthenticated...
 -> 예측 심각도: CRITICAL, 신뢰도 확률: 0.7719

설명: Information disclosure flaw where internal server paths and ...
 -> 예측 심각도: MEDIUM, 신뢰도 확률: 0.5574

설명: Denial of service issue causing high CPU usage when parsing ...
 -> 예측 심각도: HIGH, 신뢰도 확률: 0.6794

설명: Minor input validation issue in administrative panel with li...
 -> 예측 심각도: MEDIUM, 신뢰도 확률: 0.3854



## 5. 엔드투엔드 취약점 분석 및 우선순위 파이프라인 테스트
의존성 파일(`package.json`)을 직접 입력하여 전체 분석 파이프라인(`src.service.analyzer.analyze`)을 실행합니다.


In [7]:
from src.service import analyze

sample_manifest = '''
{
  "name": "sample-project",
  "dependencies": {
    "lodash": "4.17.15",
    "express": "4.16.0"
  }
}
'''

report = analyze(sample_manifest)
print(f"파일 종류: {report['file_type']}")
print(f"분석 요약: {json.dumps(report['summary'], indent=2, ensure_ascii=False)}")
print(f"\n조치 가이드 설명:\n{report['explanation']}")
print(f"설명 출처: {report['explanation_source']}")

print(f"\n발견된 취약점 수: {len(report['findings'])}건")
if report['findings']:
    print("\n상위 3개 취약점:")
    for f in report['findings'][:3]:
        print(f" - [{f['severity']}] {f['package']} {f['version']} | {f['cve_id']} | 우선순위: {f['priority']} | KEV: {f['in_kev']}")


파일 종류: package.json
분석 요약: {
  "dependency_count": 2,
  "vulnerable_dependency_count": 2,
  "vulnerability_count": 8,
  "kev_count": 0,
  "osv_failures": 0,
  "level": "high"
}

조치 가이드 설명:
라이브러리 2개 중 2개에서 취약점 8건이 발견되었습니다. 가장 급한 것은 lodash 4.17.15의 CVE-2021-23337입니다. 4.17.21 버전 이상으로 올리면 해결됩니다.
설명 출처: basic

발견된 취약점 수: 8건

상위 3개 취약점:
 - [HIGH] lodash 4.17.15 | CVE-2021-23337 | 우선순위: 48.175200000000004 | KEV: False
 - [HIGH] lodash 4.17.15 | CVE-2021-23337 | 우선순위: 39.8412 | KEV: False
 - [HIGH] lodash 4.17.15 | CVE-2020-8203 | 우선순위: 30.7972 | KEV: False


## 6. 결론 및 모델 성능 요약


In [8]:
metrics_ml_path = AI_DIR / "models" / "metrics_ml.json"
metrics_dl_path = AI_DIR / "models" / "metrics_dl.json"

if metrics_ml_path.exists():
    print("ML 모델 학습 평가 지표:")
    print(metrics_ml_path.read_text(encoding="utf-8"))

if metrics_dl_path.exists():
    print("\nDL 모델 학습 평가 지표:")
    print(metrics_dl_path.read_text(encoding="utf-8"))


ML 모델 학습 평가 지표:
{
  "rows": 17271,
  "positive": 1919,
  "auc": 0.8103
}

DL 모델 학습 평가 지표:
{
  "rows": 25000,
  "accuracy": 0.6502,
  "f1_macro": 0.5111
}
